# P1 통계 검정과 A/B 표본 수 계산

- 입력: BigQuery 집계 결과(성공 수 / 전체 수). 원본 데이터는 필요 없음
- 판정 기준(사전에 정함): **p < 0.05** 그리고 **상대 차이 10% 이상**
- 두 비율 차이 z-검정은 2×2 카이제곱 검정과 같은 결과(연속성 보정 없음)이며, 방향과 신뢰구간을 함께 보려고 z-검정을 사용

In [ ]:
# 셀 0: 두 비율 차이 검정 함수
from statsmodels.stats.proportion import proportions_ztest, confint_proportions_2indep

def compare_rates(name, success_a, n_a, success_b, n_b):
    """그룹 A와 B의 비율을 비교한다. A가 비교 대상, B가 기준."""
    rate_a = success_a / n_a
    rate_b = success_b / n_b
    z, p = proportions_ztest(count=[success_a, success_b], nobs=[n_a, n_b])
    low, high = confint_proportions_2indep(success_a, n_a, success_b, n_b, compare='diff')
    rel_diff = (rate_a / rate_b - 1) * 100
    print(f"{name}")
    print(f"  A {rate_a:.1%} vs B {rate_b:.1%} | 차이 {(rate_a-rate_b)*100:+.1f}%p "
          f"[{low*100:+.1f}, {high*100:+.1f}] | 상대 {rel_diff:+.1f}% | p = {p:.4f}")

## 1. 단계별 시즌(A) vs 비시즌(B) — 최대 도달 기준 (30번 쿼리)

결과: 앞쪽 3단계는 유의하게 하락, **결제 정보 → 구매는 차이 없음(p = 0.147)** → 1월 하락은 결제 과정이 아니라 구매 의도 감소. 결제 시작 → 결제 정보는 시즌이 더 낮음(시즌 신규 때문, 셀 5).

In [ ]:
print("=== 단계별 시즌 vs 비시즌 ===")
compare_rates("방문 → 조회",           28454, 124133, 21514, 107189)
compare_rates("조회 → 구매 의도",       9885,  28454,  4819,  21514)
compare_rates("구매 의도 → 결제 시작",  5154,   9885,  2038,   4819)
compare_rates("결제 시작 → 결제 정보",  3224,   5154,  1482,   2038)
compare_rates("결제 정보 → 구매",       2369,   3224,  1059,   1482)

## 2. H3: 비시즌 구매 의도 하락은 신규에서 더 큰가 (31번 쿼리)

두 집단의 **변화량**을 비교하는 질문이라 로지스틱 회귀의 기간 × 신규 상호작용 항으로 검정.
결과: 상호작용 계수 −0.460 (오즈비 0.631), p < 0.001 → **H3 지지**. 칸 4개에 계수 4개인 포화 모델이라 잔차 자유도 0은 정상.

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

df = pd.DataFrame({
    'period':  ['season', 'season', 'off', 'off'],
    'segment': ['new', 'returning', 'new', 'returning'],
    'success': [5689, 4196, 2604, 2215],     # 구매 의도 도달
    'n':       [18500, 9954, 15183, 6331],   # 조회 도달
})
df['rate'] = df['success'] / df['n']

model = smf.glm(
    "rate ~ C(period, Treatment('season')) * C(segment, Treatment('returning'))",
    data=df,
    family=sm.families.Binomial(),
    var_weights=df['n'],
).fit()

print(model.summary())
print(np.exp(model.params))   # 오즈비

## 3. H4: 시즌 신규 결제 정보 전환율, 기기·국가 (34번 쿼리)

결과: 모바일 +3.0% (p = 0.50), 미국 외 +2.3% (p = 0.60), 신뢰구간이 0을 포함하고 방향도 반대 → **H4 기각**

In [ ]:
compare_rates("H4 기기: 모바일 vs 데스크톱 (시즌 신규 결제 정보)", 496, 1085, 714, 1608)
compare_rates("H4 지역: 미국 외 vs 미국", 676, 1493, 556, 1256)

## 4. H4-b: 블프 주(A) vs 배송 마감 직전 주(B) (35번 쿼리)

결과: 신규 −36.9% (p < 0.0001), 재방문 −7.0% (p = 0.013이지만 크기 기준 10% 미달) → 할인 기간 이탈은 신규에 집중, **H4-b 지지**

In [ ]:
print("=== H4-b 주별 비교 ===")
compare_rates("신규: 블프 주 vs 마감 직전 주",   306, 754, 232, 361)
compare_rates("재방문: 블프 주 vs 마감 직전 주", 413, 524, 434, 512)

## 5. 시즌 신규(A) vs 시즌 재방문(B), 결제 정보 단계 (31번 쿼리)

결과: 44.8% vs 82.8%, 상대 −45.9%, p < 0.0001 → 개선 대상 확정

In [ ]:
compare_rates("시즌 결제 정보 전환율: 신규 vs 재방문", 1232, 2749, 1992, 2405)

## 6. A/B 표본 수 계산 (40번 쿼리 기준값)

- 핵심 지표: **구매 ÷ 장바구니(구매 의도) 도달**, 할인 기간 신규 기준 15.1%, 하루 223명
- (처음에는 결제 정보 ÷ 결제 시작 41.9%를 썼으나, 개선안이 장바구니에서 개입해 분모가 바뀌므로 변경. 진단 지표로만 사용)
- α = 0.05, 검정력 = 0.8 (효과가 있을 때 80% 확률로 찾아냄), 50:50 배정
- 결과: 할인 기간 21일이면 **15.1% → 18.1% (+3.0%p, 상대 +20%)** 이상의 효과를 검출 가능

In [ ]:
import math
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

baseline = 0.151   # 장바구니 도달 → 구매
daily = 223        # 할인 기간 신규 장바구니 도달 하루 평균

print("=== 효과 크기별 필요 표본 ===")
for lift in [0.01, 0.02, 0.03, 0.05]:
    target = baseline + lift
    es = proportion_effectsize(target, baseline)
    n_per_group = NormalIndPower().solve_power(
        effect_size=es, alpha=0.05, power=0.8, ratio=1, alternative='two-sided'
    )
    total = n_per_group * 2
    print(f"+{lift*100:.0f}%p (상대 +{lift/baseline*100:.0f}%) | "
          f"그룹당 {n_per_group:,.0f} | 합계 {total:,.0f} | 약 {total/daily:.1f}일")

# 할인 기간 21일로 잡을 수 있는 가장 작은 효과 (역계산)
n_21 = daily * 21 / 2
es_21 = NormalIndPower().solve_power(nobs1=n_21, alpha=0.05, power=0.8, ratio=1)
target_21 = math.sin((es_21 + 2 * math.asin(math.sqrt(baseline))) / 2) ** 2
print(f"\n=== 21일 동안 검출 가능한 최소 효과 ===")
print(f"그룹당 {n_21:,.0f} | {baseline:.1%} → {target_21:.1%} "
      f"(+{(target_21-baseline)*100:.1f}%p, 상대 +{(target_21/baseline-1)*100:.0f}%)")

## 한계
- 세션 간 독립 가정: 재방문이 섞인 비교는 p값이 실제보다 약간 작을 수 있음 (신규는 사람당 한 세션)
- 사전에 정한 가설만 검정. 결론에 쓴 결과는 p < 0.0001이거나 p > 0.5로 판정이 흔들리지 않음
- 검정은 "차이가 우연이 아니다"까지. 이탈 원인(최종 금액 확인)은 해석이며 A/B로 검증할 가설